In [117]:
import requests
import pandas as pd

API_URL = f"https://tabular-api.data.gouv.fr/api/resources/77ed6b2f-c48f-4037-8479-50af74fa5c7a/data/"
df_election = pd.read_csv('src/election.csv', sep=';')

all_data = []
page = 1

while True:
    response = requests.get(API_URL, params={"page": page, "page_size": 200})
    result = response.json()
    all_data.extend(result["data"])
    
    if result["links"]["next"] is None:
        break
    page += 1

df = pd.DataFrame(all_data)
display(df)

/tmp/ipykernel_9185/3892326077.py:5: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  df_election = pd.read_csv('src/election.csv', sep=';')


,__id,Résultats par communes,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,...,Unnamed: 85,Unnamed: 86,Unnamed: 87,Unnamed: 88,Unnamed: 89,Unnamed: 90,Unnamed: 91,Unnamed: 92,Unnamed: 93,Unnamed: 94
0,1,None,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
1,2,None,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
2,3,Code du département,Libellé du département,Code de la commune,Libellé de la commune,Inscrits,Abstentions,% Abs/Ins,Votants,% Vot/Ins,...,Voix,% Voix/Ins,% Voix/Exp,N°Panneau,Sexe,Nom,Prénom,Voix,% Voix/Ins,% Voix/Exp
3,4,1.0,Ain,1.0,L'Abergement-Clémenciat,598.0,92.0,15.38,506.0,84.62,...,2.0,0.33,0.4,8.0,M,LASSALLE,Jean,2.0,0.33,0.4
4,5,1.0,Ain,2.0,L'Abergement-de-Varey,209.0,25.0,11.96,184.0,88.04,...,0.0,0.0,0.0,8.0,M,LASSALLE,Jean,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35717,35718,ZZ,Français établis hors de France,226.0,Wuhan,369.0,129.0,34.96,240.0,65.04,...,0.0,0.0,0.0,8.0,M,LASSALLE,Jean,0.0,0.0,0.0
35718,35719,ZZ,Français établis hors de France,227.0,Yaounde,1514.0,800.0,52.84,714.0,47.16,...,0.0,0.0,0.0,7.0,M,CHEMINADE,Jacques,0.0,0.0,0.0
35719,35720,ZZ,Français établis hors de France,228.0,Zagreb,655.0,401.0,61.22,254.0,38.78,...,0.0,0.0,0.0,7.0,M,CHEMINADE,Jacques,0.0,0.0,0.0
35720,35721,ZZ,Français établis hors de France,229.0,Zurich,21477.0,11129.0,51.82,10348.0,48.18,...,27.0,0.13,0.26,7.0,M,CHEMINADE,Jacques,18.0,0.08,0.18


In [118]:
new_cols = list(df.columns[:2]) + list(df.iloc[2, 2:])
df.columns = new_cols
df = df.iloc[3:].reset_index(drop=True)

In [119]:
df.columns

Index(['__id', 'Résultats par communes ', 'Libellé du département',
       'Code de la commune', 'Libellé de la commune', 'Inscrits',
       'Abstentions', '% Abs/Ins', 'Votants', '% Vot/Ins', 'Blancs',
       '% Blancs/Ins', '% Blancs/Vot', 'Nuls', '% Nuls/Ins', '% Nuls/Vot',
       'Exprimés', '% Exp/Ins', '% Exp/Vot', 'N°Panneau', 'Sexe', 'Nom',
       'Prénom', 'Voix', '% Voix/Ins', '% Voix/Exp', 'N°Panneau', 'Sexe',
       'Nom', 'Prénom', 'Voix', '% Voix/Ins', '% Voix/Exp', 'N°Panneau',
       'Sexe', 'Nom', 'Prénom', 'Voix', '% Voix/Ins', '% Voix/Exp',
       'N°Panneau', 'Sexe', 'Nom', 'Prénom', 'Voix', '% Voix/Ins',
       '% Voix/Exp', 'N°Panneau', 'Sexe', 'Nom', 'Prénom', 'Voix',
       '% Voix/Ins', '% Voix/Exp', 'N°Panneau', 'Sexe', 'Nom', 'Prénom',
       'Voix', '% Voix/Ins', '% Voix/Exp', 'N°Panneau', 'Sexe', 'Nom',
       'Prénom', 'Voix', '% Voix/Ins', '% Voix/Exp', 'N°Panneau', 'Sexe',
       'Nom', 'Prénom', 'Voix', '% Voix/Ins', '% Voix/Exp', 'N°Panneau',
       'S

In [120]:
CLASSIFICATION = {
    "ARTHAUD":       "gauche", 
    "POUTOU":        "gauche",
    "HAMON":         "gauche",
    "MÉLENCHON":     "gauche",
    "MACRON":        "centre",
    "FILLON":        "droite",
    "LE PEN":        "droite",
    "DUPONT-AIGNAN": "droite",
    "ASSELINEAU":    "droite",
    "LASSALLE":      "centre",
    "CHEMINADE":     "droite",
}

all_cols = list(df.columns)
nom_indices = [i for i, c in enumerate(all_cols) if c == "Nom"]
pct_indices = [i for i, c in enumerate(all_cols) if c == "% Voix/Exp"]

df["% gauche/Exp"] = 0.0
df["% centre/Exp"] = 0.0
df["% droite/Exp"] = 0.0

for nom_idx, pct_idx in zip(nom_indices, pct_indices):
    nom = df.iloc[0, nom_idx]
    bloc = CLASSIFICATION.get(nom, "autre")
    pct = pd.to_numeric(df.iloc[:, pct_idx], errors="coerce").fillna(0)
    
    if bloc == "gauche":
        df["% gauche/Exp"] = df["% gauche/Exp"] + pct
    elif bloc == "centre":
        df["% centre/Exp"] = df["% centre/Exp"] + pct
    elif bloc == "droite":
        df["% droite/Exp"] = df["% droite/Exp"] + pct

df_out = df[["Libellé de la commune", "% gauche/Exp", "% centre/Exp", "% droite/Exp"]].round(2)

display(df_out) 

,Libellé de la commune,% gauche/Exp,% centre/Exp,% droite/Exp
0,L'Abergement-Clémenciat,19.40,24.44,56.15
1,L'Abergement-de-Varey,23.87,21.02,55.12
2,Ambérieu-en-Bugey,24.16,21.96,53.87
3,Ambérieux-en-Dombes,19.18,21.11,59.70
4,Ambléon,24.68,23.38,51.95
...,...,...,...,...
35714,Wuhan,8.89,26.27,64.84
35715,Yaounde,11.11,27.71,61.17
35716,Zagreb,15.54,25.90,58.56
35717,Zurich,9.07,28.42,62.51


In [121]:
df_out = df[[
    "Libellé de la commune",
    "Inscrits",
    "Abstentions",
    "% Abs/Ins",
    "Votants",
    "% Vot/Ins",
    "Blancs",
    "% Blancs/Ins",
    "% Blancs/Vot",
    "Nuls",
    "% Nuls/Ins",
    "% Nuls/Vot",
    "Exprimés",
    "% Exp/Ins",
    "% Exp/Vot",
]].copy()

df_out["% gauche/Exp"] = gauche.round(2)
df_out["% centre/Exp"] = centre.round(2)
df_out["% droite/Exp"] = droite.round(2)
df_out['Année'] = 2017
display(df_out)

,Libellé de la commune,Inscrits,Abstentions,% Abs/Ins,Votants,% Vot/Ins,Blancs,% Blancs/Ins,% Blancs/Vot,Nuls,% Nuls/Ins,% Nuls/Vot,Exprimés,% Exp/Ins,% Exp/Vot,% gauche/Exp,% centre/Exp,% droite/Exp,Année
0,L'Abergement-Clémenciat,598.0,92.0,15.38,506.0,84.62,2.0,0.33,0.4,9.0,1.51,1.78,495.0,82.78,97.83,19.40,24.44,56.15,2017
1,L'Abergement-de-Varey,209.0,25.0,11.96,184.0,88.04,6.0,2.87,3.26,2.0,0.96,1.09,176.0,84.21,95.65,23.87,21.02,55.12,2017
2,Ambérieu-en-Bugey,8586.0,1962.0,22.85,6624.0,77.15,114.0,1.33,1.72,58.0,0.68,0.88,6452.0,75.15,97.4,24.16,21.96,53.87,2017
3,Ambérieux-en-Dombes,1172.0,215.0,18.34,957.0,81.66,21.0,1.79,2.19,3.0,0.26,0.31,933.0,79.61,97.49,19.18,21.11,59.70,2017
4,Ambléon,99.0,20.0,20.2,79.0,79.8,2.0,2.02,2.53,0.0,0.0,0.0,77.0,77.78,97.47,24.68,23.38,51.95,2017
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35714,Wuhan,369.0,129.0,34.96,240.0,65.04,2.0,0.54,0.83,2.0,0.54,0.83,236.0,63.96,98.33,8.89,26.27,64.84,2017
35715,Yaounde,1514.0,800.0,52.84,714.0,47.16,3.0,0.2,0.42,0.0,0.0,0.0,711.0,46.96,99.58,11.11,27.71,61.17,2017
35716,Zagreb,655.0,401.0,61.22,254.0,38.78,2.0,0.31,0.79,1.0,0.15,0.39,251.0,38.32,98.82,15.54,25.90,58.56,2017
35717,Zurich,21477.0,11129.0,51.82,10348.0,48.18,47.0,0.22,0.45,58.0,0.27,0.56,10243.0,47.69,98.99,9.07,28.42,62.51,2017


In [122]:
df_out = df_out.merge(
    df_election[["Libellé de la commune", "Code_INSEE"]],
    on="Libellé de la commune",
    how="left"
)

In [123]:
df_out["Résultat"] = df_out[["% gauche/Exp", "% centre/Exp", "% droite/Exp"]].idxmax(axis=1).str.replace("% ", "").str.replace("/Exp", "")

In [124]:
display(df_out)

,Libellé de la commune,Inscrits,Abstentions,% Abs/Ins,Votants,% Vot/Ins,Blancs,% Blancs/Ins,% Blancs/Vot,Nuls,...,% Nuls/Vot,Exprimés,% Exp/Ins,% Exp/Vot,% gauche/Exp,% centre/Exp,% droite/Exp,Année,Code_INSEE,Résultat
0,L'Abergement-Clémenciat,598.0,92.0,15.38,506.0,84.62,2.0,0.33,0.4,9.0,...,1.78,495.0,82.78,97.83,19.40,24.44,56.15,2017,01001,droite
1,L'Abergement-Clémenciat,598.0,92.0,15.38,506.0,84.62,2.0,0.33,0.4,9.0,...,1.78,495.0,82.78,97.83,19.40,24.44,56.15,2017,01001,droite
2,L'Abergement-de-Varey,209.0,25.0,11.96,184.0,88.04,6.0,2.87,3.26,2.0,...,1.09,176.0,84.21,95.65,23.87,21.02,55.12,2017,01002,droite
3,L'Abergement-de-Varey,209.0,25.0,11.96,184.0,88.04,6.0,2.87,3.26,2.0,...,1.09,176.0,84.21,95.65,23.87,21.02,55.12,2017,01002,droite
4,Ambérieu-en-Bugey,8586.0,1962.0,22.85,6624.0,77.15,114.0,1.33,1.72,58.0,...,0.88,6452.0,75.15,97.4,24.16,21.96,53.87,2017,01004,droite
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
86438,Wuhan,369.0,129.0,34.96,240.0,65.04,2.0,0.54,0.83,2.0,...,0.83,236.0,63.96,98.33,8.89,26.27,64.84,2017,ZZ226,droite
86439,Yaounde,1514.0,800.0,52.84,714.0,47.16,3.0,0.2,0.42,0.0,...,0.0,711.0,46.96,99.58,11.11,27.71,61.17,2017,ZZ227,droite
86440,Zagreb,655.0,401.0,61.22,254.0,38.78,2.0,0.31,0.79,1.0,...,0.39,251.0,38.32,98.82,15.54,25.90,58.56,2017,ZZ228,droite
86441,Zurich,21477.0,11129.0,51.82,10348.0,48.18,47.0,0.22,0.45,58.0,...,0.56,10243.0,47.69,98.99,9.07,28.42,62.51,2017,ZZ229,droite


In [125]:

df_out = df_out.dropna()
display(df_out)

,Libellé de la commune,Inscrits,Abstentions,% Abs/Ins,Votants,% Vot/Ins,Blancs,% Blancs/Ins,% Blancs/Vot,Nuls,...,% Nuls/Vot,Exprimés,% Exp/Ins,% Exp/Vot,% gauche/Exp,% centre/Exp,% droite/Exp,Année,Code_INSEE,Résultat
0,L'Abergement-Clémenciat,598.0,92.0,15.38,506.0,84.62,2.0,0.33,0.4,9.0,...,1.78,495.0,82.78,97.83,19.40,24.44,56.15,2017,01001,droite
1,L'Abergement-Clémenciat,598.0,92.0,15.38,506.0,84.62,2.0,0.33,0.4,9.0,...,1.78,495.0,82.78,97.83,19.40,24.44,56.15,2017,01001,droite
2,L'Abergement-de-Varey,209.0,25.0,11.96,184.0,88.04,6.0,2.87,3.26,2.0,...,1.09,176.0,84.21,95.65,23.87,21.02,55.12,2017,01002,droite
3,L'Abergement-de-Varey,209.0,25.0,11.96,184.0,88.04,6.0,2.87,3.26,2.0,...,1.09,176.0,84.21,95.65,23.87,21.02,55.12,2017,01002,droite
4,Ambérieu-en-Bugey,8586.0,1962.0,22.85,6624.0,77.15,114.0,1.33,1.72,58.0,...,0.88,6452.0,75.15,97.4,24.16,21.96,53.87,2017,01004,droite
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
86436,Wellington,2834.0,1622.0,57.23,1212.0,42.77,16.0,0.56,1.32,7.0,...,0.58,1189.0,41.95,98.1,16.07,24.13,59.80,2017,ZZ224,droite
86438,Wuhan,369.0,129.0,34.96,240.0,65.04,2.0,0.54,0.83,2.0,...,0.83,236.0,63.96,98.33,8.89,26.27,64.84,2017,ZZ226,droite
86439,Yaounde,1514.0,800.0,52.84,714.0,47.16,3.0,0.2,0.42,0.0,...,0.0,711.0,46.96,99.58,11.11,27.71,61.17,2017,ZZ227,droite
86440,Zagreb,655.0,401.0,61.22,254.0,38.78,2.0,0.31,0.79,1.0,...,0.39,251.0,38.32,98.82,15.54,25.90,58.56,2017,ZZ228,droite


In [126]:
df_out.drop_duplicates(subset=['Code_INSEE'])

,Libellé de la commune,Inscrits,Abstentions,% Abs/Ins,Votants,% Vot/Ins,Blancs,% Blancs/Ins,% Blancs/Vot,Nuls,...,% Nuls/Vot,Exprimés,% Exp/Ins,% Exp/Vot,% gauche/Exp,% centre/Exp,% droite/Exp,Année,Code_INSEE,Résultat
0,L'Abergement-Clémenciat,598.0,92.0,15.38,506.0,84.62,2.0,0.33,0.4,9.0,...,1.78,495.0,82.78,97.83,19.40,24.44,56.15,2017,01001,droite
2,L'Abergement-de-Varey,209.0,25.0,11.96,184.0,88.04,6.0,2.87,3.26,2.0,...,1.09,176.0,84.21,95.65,23.87,21.02,55.12,2017,01002,droite
4,Ambérieu-en-Bugey,8586.0,1962.0,22.85,6624.0,77.15,114.0,1.33,1.72,58.0,...,0.88,6452.0,75.15,97.4,24.16,21.96,53.87,2017,01004,droite
6,Ambérieux-en-Dombes,1172.0,215.0,18.34,957.0,81.66,21.0,1.79,2.19,3.0,...,0.31,933.0,79.61,97.49,19.18,21.11,59.70,2017,01005,droite
8,Ambléon,99.0,20.0,20.2,79.0,79.8,2.0,2.02,2.53,0.0,...,0.0,77.0,77.78,97.47,24.68,23.38,51.95,2017,01006,droite
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
86436,Wellington,2834.0,1622.0,57.23,1212.0,42.77,16.0,0.56,1.32,7.0,...,0.58,1189.0,41.95,98.1,16.07,24.13,59.80,2017,ZZ224,droite
86438,Wuhan,369.0,129.0,34.96,240.0,65.04,2.0,0.54,0.83,2.0,...,0.83,236.0,63.96,98.33,8.89,26.27,64.84,2017,ZZ226,droite
86439,Yaounde,1514.0,800.0,52.84,714.0,47.16,3.0,0.2,0.42,0.0,...,0.0,711.0,46.96,99.58,11.11,27.71,61.17,2017,ZZ227,droite
86440,Zagreb,655.0,401.0,61.22,254.0,38.78,2.0,0.31,0.79,1.0,...,0.39,251.0,38.32,98.82,15.54,25.90,58.56,2017,ZZ228,droite


In [129]:
df_election = pd.concat([df_election, df_out], ignore_index=True)
df_election['Code_INSEE'] = df_election['Code_INSEE'].astype(str)
df_election = df_election.sort_values(by=['Code_INSEE']).reset_index(drop=True)
df_election = df_election.drop_duplicates(subset=['Code_INSEE', 'Année']).reset_index(drop=True)
display(df_election)

df_election.to_csv('src/election.csv', index=False, sep=';', encoding='utf-8')

,Année,Code_INSEE,Libellé de la commune,Inscrits,Abstentions,% Abs/Ins,Votants,% Vot/Ins,Blancs,% Blancs/Ins,...,Nuls,% Nuls/Ins,% Nuls/Vot,Exprimés,% Exp/Ins,% Exp/Vot,% gauche/Exp,% centre/Exp,% droite/Exp,Résultat
0,2022,01001,L'Abergement-Clémenciat,645,108,16.74,537,83.26,16,2.48,...,1,0.16,0.19,520,80.62,96.83,21.730000,32.310000,45.960000,droite
1,2017,01001,L'Abergement-Clémenciat,598.0,92.0,15.38,506.0,84.62,2.0,0.33,...,9.0,1.51,1.78,495.0,82.78,97.83,19.400000,24.440000,56.150000,droite
2,2012,01001,L'Abergement-Clémenciat,592,84,14.189189,508,85.810811,9,1.52027,...,0,0.0,0.0,499,84.290541,98.228346,30.861723,10.821643,58.316633,droite
3,2017,01002,L'Abergement-de-Varey,209.0,25.0,11.96,184.0,88.04,6.0,2.87,...,2.0,0.96,1.09,176.0,84.21,95.65,23.870000,21.020000,55.120000,droite
4,2022,01002,L'Abergement-de-Varey,213,38,17.84,175,82.16,3,1.41,...,1,0.47,0.57,171,80.28,97.71,38.600000,35.090000,26.320000,gauche
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
105141,2022,ZZ229,Zurich,24868,14101,56.7,10767,43.3,40,0.16,...,31,0.12,0.29,10696,43.01,99.34,26.670000,54.920000,18.410000,centre
105142,2022,ZZ231,Taipei,1709,942,55.12,767,44.88,8,0.47,...,2,0.12,0.26,757,44.29,98.7,37.380000,39.230000,23.380000,centre
105143,2022,ZZ233,Nour-Soultan,117,64,54.7,53,45.3,2,1.71,...,0,0.0,0.0,51,43.59,96.23,27.450000,50.980000,21.570000,centre
105144,2022,ZZ234,Monterrey,713,553,77.56,160,22.44,0,0.0,...,2,0.28,1.25,158,22.16,98.75,20.890000,61.390000,17.720000,centre
